# بازپیاده‌سازی کامل استراتژی نگهداری پویای FD001

این نوت‌بوک برای Google Colab طراحی شده و دو منبع را به‌صورت شفاف ترکیب می‌کند:

1. **مدل پیش‌بینی:** تنظیمات واقعی کد منتشرشده نویسنده (پنجره ۲۵، سقف RUL برابر ۱۲۵، Min-Max، یک Conv1D، BiLSTM با ۶۴ واحد، دو Dropout، ۶۴ دوره و دسته ۲۵۶).
2. **استراتژی نگهداری:** روابط (۶) تا (۱۴) مقاله Wang et al. (2024)، شامل تصمیم نگهداری، متغیر خطا، مأموریت، سفارش قطعه، موجودی و نرخ هزینه.

تقسیم موتورهای **۱ تا ۸۰ برای آموزش** و **۸۱ تا ۱۰۰ برای ارزیابی چرخه‌ای** دقیقاً مطابق بخش 4.3 مقاله است. مقاله برای شکل‌های استراتژی از میانگین ۱۰۰ آزمایش استفاده کرده است؛ برای پایان‌نامه و زمان اجرای عملی، مقدار پیش‌فرض `N_RUNS=10` قرار داده شده و قابل تغییر به ۱۰۰ است.

> نکته روش‌شناختی: مقاله مقدار δ را از خطای نمونه‌های آزمون دارای RUL کمتر از ۳۰ محاسبه می‌کند. این نوت‌بوک برای وفاداری به مقاله همین روش را اجرا می‌کند و آن را در خروجی به‌عنوان محدودیت ثبت می‌کند.

In [ ]:
# سلول 1: کتابخانه‌ها، GPU و تنظیمات
import os, gc, time, random, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from scipy import interpolate
from scipy.stats import gamma as gamma_dist
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Conv1D, Dropout, Bidirectional, LSTM, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from google.colab import files

warnings.filterwarnings("ignore")

# تنظیمات واقعی کد نویسنده
MAX_RUL = 125.0
WINDOW_SIZE = 25
INPUT_DIMS = 14
CNN_FILTERS = 64
KERNEL_SIZE = 3
BILSTM_UNITS = 64
DROPOUT_RATE = 0.3
LEARNING_RATE = 0.001
BATCH_SIZE = 256
EPOCHS = 64

# اجرای نهایی پایان‌نامه؛ مقاله در بخش استراتژی از 100 آزمایش استفاده کرده است.
N_RUNS = 10
SEEDS = [11, 22, 33, 44, 55, 66, 77, 88, 99, 110]
if N_RUNS > len(SEEDS):
    SEEDS = [11 * i for i in range(1, N_RUNS + 1)]

# پارامترهای مقاله
GAMMA_RATE = 2.0
GAMMA_SHAPE = 3
MISSION_CYCLE = 3.770
LEAD_TIME = 20.0
INSPECTION_INTERVAL = 5
CP = 100.0
CF = 500.0  # مقدار استفاده‌شده در بخش مقایسه هزینه مقاله
COS = 10.0
RELIABILITY_THRESHOLD = 0.8

OUTPUT_DIR = Path("/content/fd001_dynamic_maintenance_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("N_RUNS:", N_RUNS)

## بارگذاری داده‌ها

پس از اجرای سلول بعد، سه فایل `train_FD001.txt`، `test_FD001.txt` و `RUL_FD001.txt` را هم‌زمان انتخاب کن. برای استراتژی چرخه‌ای، مسیر کامل موتورهای فایل آموزش استفاده می‌شود؛ فایل‌های آزمون رسمی فقط برای کنترل ابعاد و حفظ مجموعه کامل پروژه بارگذاری می‌شوند.

In [ ]:
# سلول 2: بارگذاری سه فایل FD001
uploaded = files.upload()

def find_uploaded(prefix):
    matches = [name for name in uploaded if name.lower().startswith(prefix.lower())]
    if not matches:
        raise FileNotFoundError(f"File starting with {prefix} was not uploaded.")
    return matches[0]

TRAIN_FILE = find_uploaded("train_FD001")
TEST_FILE = find_uploaded("test_FD001")
RUL_FILE = find_uploaded("RUL_FD001")

train_raw = np.loadtxt(TRAIN_FILE)
official_test_raw = np.loadtxt(TEST_FILE)
official_test_rul = np.loadtxt(RUL_FILE).reshape(-1)

assert train_raw.shape == (20631, 26), train_raw.shape
assert official_test_raw.shape == (13096, 26), official_test_raw.shape
assert official_test_rul.shape == (100,), official_test_rul.shape
print("Files and dimensions are correct.")

## پیش‌پردازش بدون نشت اطلاعات

موتورهای ۱ تا ۸۰ مبنای برازش Min-Max هستند و همان تبدیل روی موتورهای ۸۱ تا ۱۰۰ اعمال می‌شود. ستون‌های ثابت حذف می‌شوند؛ فهرست حذف دقیقاً مطابق کد نویسنده است و پس از نگه‌داشتن شناسه و چرخه، ۱۴ حسگر باقی می‌ماند.

In [ ]:
# سلول 3: تقسیم موتورمحور، مقیاس‌بندی و انتخاب حسگرها
TRAIN_ENGINE_IDS = np.arange(1, 81)
STRATEGY_ENGINE_IDS = np.arange(81, 101)
DELETE_COLUMNS = [2, 3, 4, 5, 9, 10, 14, 20, 22, 23]

train_mask = np.isin(train_raw[:, 0].astype(int), TRAIN_ENGINE_IDS)
strategy_mask = np.isin(train_raw[:, 0].astype(int), STRATEGY_ENGINE_IDS)

raw_80 = train_raw[train_mask].copy()
raw_20 = train_raw[strategy_mask].copy()

# شناسه و چرخه مقیاس نمی‌شوند؛ پارامترهای scaler فقط از موتورهای 1 تا 80 می‌آیند.
scaler = MinMaxScaler()
raw_80[:, 2:] = scaler.fit_transform(raw_80[:, 2:])
raw_20[:, 2:] = scaler.transform(raw_20[:, 2:])

data_80 = np.delete(raw_80, DELETE_COLUMNS, axis=1)
data_20 = np.delete(raw_20, DELETE_COLUMNS, axis=1)

assert data_80.shape[1] == 16  # engine, cycle, 14 sensors
assert data_20.shape[1] == 16
print("Training engines:", np.unique(data_80[:, 0].astype(int)))
print("Strategy engines:", np.unique(data_20[:, 0].astype(int)))
print("Number of sensor inputs:", data_80.shape[1] - 2)

In [ ]:
# سلول 4: ساخت پنجره‌های آموزش و پیش‌بینی چرخه‌ای
def make_training_windows(data, window_size=25, max_rul=125.0):
    X, y = [], []
    for engine_id in np.unique(data[:, 0].astype(int)):
        engine = data[data[:, 0].astype(int) == engine_id]
        engine = engine[np.argsort(engine[:, 1])]
        lifetime = int(engine[-1, 1])
        for start in range(len(engine) - window_size + 1):
            end = start + window_size
            end_cycle = int(engine[end - 1, 1])
            rul = min(lifetime - end_cycle, max_rul)
            X.append(engine[start:end, 2:])
            y.append(rul / max_rul)
    return np.asarray(X, np.float32), np.asarray(y, np.float32)


def make_cycle_windows(data, window_size=25, max_rul=125.0):
    X, engine_ids, cycles, true_rul, capped_rul, lifetimes = [], [], [], [], [], []
    for engine_id in np.unique(data[:, 0].astype(int)):
        engine = data[data[:, 0].astype(int) == engine_id]
        engine = engine[np.argsort(engine[:, 1])]
        lifetime = int(engine[-1, 1])
        for end in range(window_size, len(engine) + 1):
            cycle = int(engine[end - 1, 1])
            rul = float(lifetime - cycle)
            X.append(engine[end-window_size:end, 2:])
            engine_ids.append(engine_id)
            cycles.append(cycle)
            true_rul.append(rul)
            capped_rul.append(min(rul, max_rul))
            lifetimes.append(lifetime)
    return tuple(np.asarray(v) for v in (X, engine_ids, cycles, true_rul, capped_rul, lifetimes))


trainData, trainTarget = make_training_windows(data_80, WINDOW_SIZE, MAX_RUL)
(strategyData, test_engine_ids, test_cycles, test_true_rul,
 test_true_rul_capped, test_lifetimes) = make_cycle_windows(data_20, WINDOW_SIZE, MAX_RUL)

strategyData = strategyData.astype(np.float32)
test_engine_ids = test_engine_ids.astype(int)
test_cycles = test_cycles.astype(int)
test_lifetimes = test_lifetimes.astype(int)

print("trainData:", trainData.shape)
print("strategyData:", strategyData.shape)
print("strategy engines:", len(np.unique(test_engine_ids)))
assert trainData.shape[1:] == (25, 14)
assert strategyData.shape[1:] == (25, 14)

In [ ]:
# سلول 5: مدل CNN-BiLSTM مطابق تنظیمات واقعی کد نویسنده
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def build_model(dropout_rate=DROPOUT_RATE):
    inputs = Input(shape=(WINDOW_SIZE, INPUT_DIMS), name="sensor_input")
    x = Conv1D(CNN_FILTERS, KERNEL_SIZE, activation="relu", padding="same", name="conv1d")(inputs)
    x = Dropout(dropout_rate, name="dropout_after_cnn")(x)
    x = Bidirectional(LSTM(BILSTM_UNITS, return_sequences=True), name="bilstm")(x)
    x = Dropout(dropout_rate, name="dropout_after_bilstm")(x)
    x = Flatten(name="flatten")(x)
    outputs = Dense(1, activation="linear", name="rul_output")(x)
    model = Model(inputs, outputs, name="CNN_BiLSTM_author_code")
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="mse")
    return model

sample_model = build_model()
sample_model.summary()
del sample_model
tf.keras.backend.clear_session()

## آموزش مستقل مدل‌ها

پس از هر اجرا، پیش‌بینی‌ها و خلاصه نتایج موقت ذخیره می‌شوند. اگر فقط قصد آزمون کد را داری، در سلول اول `N_RUNS=1` قرار بده. برای نتایج پایان‌نامه مقدار ۱۰ و برای تقلید دقیق تعداد ensemble بخش استراتژی مقاله مقدار ۱۰۰ لازم است.

In [ ]:
# سلول 6: ده اجرای مستقل و پیش‌بینی تجمیعی
run_predictions, run_rmse_values, run_training_times, run_final_losses = [], [], [], []
history_losses = []

for run_idx, seed in enumerate(SEEDS[:N_RUNS], start=1):
    set_seed(seed)
    tf.keras.backend.clear_session()
    gc.collect()
    model = build_model(DROPOUT_RATE)
    started = time.time()
    history = model.fit(
        trainData, trainTarget,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        shuffle=True,
        verbose=0
    )
    elapsed = time.time() - started
    pred = model.predict(strategyData, batch_size=BATCH_SIZE, verbose=0).reshape(-1) * MAX_RUL
    pred = np.maximum(pred, 0.0)
    rmse = float(np.sqrt(mean_squared_error(test_true_rul_capped, pred)))

    run_predictions.append(pred.astype(np.float32))
    run_rmse_values.append(rmse)
    run_training_times.append(elapsed)
    run_final_losses.append(float(history.history["loss"][-1]))
    history_losses.append(np.asarray(history.history["loss"], dtype=float))

    print(f"Run {run_idx}/{N_RUNS} | RMSE={rmse:.4f} | time={elapsed:.1f}s")

    np.savez_compressed(
        OUTPUT_DIR / "strategy_training_checkpoint.npz",
        run_predictions=np.asarray(run_predictions),
        run_rmse_values=np.asarray(run_rmse_values),
        run_training_times=np.asarray(run_training_times),
        history_losses=np.asarray(history_losses),
        test_engine_ids=test_engine_ids,
        test_cycles=test_cycles,
        test_true_rul=test_true_rul,
        test_true_rul_capped=test_true_rul_capped,
        test_lifetimes=test_lifetimes,
    )
    del model, history
    tf.keras.backend.clear_session()
    gc.collect()

run_predictions = np.asarray(run_predictions, dtype=float)
ensemble_prediction = np.mean(run_predictions, axis=0)
ensemble_rmse = float(np.sqrt(mean_squared_error(test_true_rul_capped, ensemble_prediction)))

run_results_df = pd.DataFrame({
    "run": np.arange(1, N_RUNS + 1),
    "seed": SEEDS[:N_RUNS],
    "rmse": run_rmse_values,
    "final_loss": run_final_losses,
    "training_time_seconds": run_training_times,
})
run_results_df.to_csv(OUTPUT_DIR / "01_training_run_results.csv", index=False)

training_summary_df = pd.DataFrame([{
    "number_of_runs": N_RUNS,
    "mean_individual_rmse": np.mean(run_rmse_values),
    "std_individual_rmse": np.std(run_rmse_values, ddof=1) if N_RUNS > 1 else 0.0,
    "ensemble_rmse": ensemble_rmse,
    "dropout": DROPOUT_RATE,
}])
training_summary_df.to_csv(OUTPUT_DIR / "02_training_summary.csv", index=False)
display(training_summary_df)

In [ ]:
# سلول 7: نمودار میانگین Loss اجراها
history_matrix = np.asarray(history_losses)
mean_loss = history_matrix.mean(axis=0)
std_loss = history_matrix.std(axis=0)
epochs_axis = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_axis, mean_loss, color="#005F73", linewidth=2, label="Mean training loss")
plt.fill_between(epochs_axis, mean_loss-std_loss, mean_loss+std_loss, color="#94D2BD", alpha=0.35, label="±1 SD")
plt.xlabel("Epoch"); plt.ylabel("MSE loss"); plt.title("CNN-BiLSTM training loss")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_training_loss.png", dpi=300, bbox_inches="tight")
plt.show()

## محاسبه δ و چرخه مأموریت

مطابق رابطه (۷)، δ میانگین قدرمطلق خطا برای نمونه‌هایی است که RUL واقعی آن‌ها کمتر از ۳۰ است. مطابق رابطه (۱۴)، چرخه مأموریت از توزیع گاما با نرخ λ=۲ و شکل k=۳ انتخاب می‌شود؛ مقدار پوشش ۹۸٪ تقریباً ۳٫۷۷ چرخه است.

In [ ]:
# سلول 8: δ، آستانه‌ها و نمودار توزیع گاما
strategy_prediction = np.maximum(ensemble_prediction, 0.0)
delta_mask = test_true_rul < 30
delta_errors = np.abs(strategy_prediction[delta_mask] - test_true_rul[delta_mask])
delta_value = float(np.mean(delta_errors))
maintenance_threshold = MISSION_CYCLE + delta_value
order_threshold = LEAD_TIME + MISSION_CYCLE + delta_value

delta_details_df = pd.DataFrame({
    "engine_id": test_engine_ids[delta_mask],
    "cycle": test_cycles[delta_mask],
    "actual_rul": test_true_rul[delta_mask],
    "predicted_rul": strategy_prediction[delta_mask],
    "absolute_error": delta_errors,
})
delta_details_df.to_csv(OUTPUT_DIR / "03_delta_details.csv", index=False)

parameter_df = pd.DataFrame([
    ["delta", "Mean absolute error for true RUL < 30", delta_value],
    ["mission_cycle", "Gamma 98% reference for k=3, lambda=2", MISSION_CYCLE],
    ["lead_time", "Spare-part delivery cycle Q", LEAD_TIME],
    ["inspection_interval", "Inspection interval", INSPECTION_INTERVAL],
    ["maintenance_threshold", "Delta t + delta", maintenance_threshold],
    ["order_threshold", "Q + Delta t + delta", order_threshold],
    ["Cp", "Predictive maintenance cost", CP],
    ["Cf", "Corrective maintenance cost", CF],
    ["Cos", "Out-of-stock cost", COS],
], columns=["parameter", "description", "value"])
parameter_df.to_csv(OUTPUT_DIR / "04_strategy_parameters.csv", index=False)
display(parameter_df)

x = np.linspace(0, 8, 700)
plt.figure(figsize=(9, 5.5))
colors = ["#0077B6", "#F77F00", "#2A9D8F", "#7B2CBF"]
for k, color in zip([2, 3, 4, 5], colors):
    y = gamma_dist.pdf(x, a=k, scale=1/GAMMA_RATE)
    q98 = gamma_dist.ppf(0.98, a=k, scale=1/GAMMA_RATE)
    plt.plot(x, y, linewidth=2, color=color, label=f"k={k}, lambda=2, q98={q98:.2f}")
    plt.scatter([q98], [gamma_dist.pdf(q98, a=k, scale=1/GAMMA_RATE)], color=color, s=30)
plt.axvline(MISSION_CYCLE, color="black", linestyle="--", alpha=0.7, label="Selected Delta t=3.770")
plt.xlabel("Mission cycle"); plt.ylabel("Probability density")
plt.title("Gamma distribution of mission-cycle duration")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_gamma_mission_cycle.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# سلول 9: تابع کامل تصمیم سفارش، موجودی و نگهداری
prediction_df = pd.DataFrame({
    "engine_id": test_engine_ids,
    "cycle": test_cycles,
    "true_rul": test_true_rul,
    "true_rul_capped": test_true_rul_capped,
    "predicted_rul": strategy_prediction,
    "lifetime": test_lifetimes,
})


def run_strategy(pred_df, mission_cycle, delta, lead_time=20.0, inspection_interval=5):
    maint_threshold = mission_cycle + delta
    ord_threshold = lead_time + mission_cycle + delta
    decisions, summaries = [], []

    for engine_id in sorted(pred_df.engine_id.unique()):
        e = pred_df[pred_df.engine_id == engine_id].sort_values("cycle").reset_index(drop=True)
        lifetime = int(e.lifetime.iloc[0])
        inspections = e[e.cycle.astype(int) % int(inspection_interval) == 0]
        ordered, spare = 0, 0
        order_cycle = arrival_cycle = maintenance_cycle = None
        maintenance_type = None
        final_shortage = 0
        completed = False

        for _, row in inspections.iterrows():
            cycle = int(row.cycle); true_rul = float(row.true_rul)
            pred = max(float(row.predicted_rul), 0.0)
            order_event = maintenance_event = shortage = 0

            if ordered and not spare and arrival_cycle is not None and cycle >= arrival_cycle:
                spare = 1

            if cycle >= lifetime or true_rul <= 0:
                maintenance_event = 1; maintenance_cycle = lifetime
                maintenance_type = "Corrective maintenance"
                shortage = int(spare == 0); final_shortage = shortage
                decision = "Corrective maintenance" + (" with out-of-stock" if shortage else "")
                n_star, gamma_star = 0, np.nan
                decisions.append(locals_to_decision(engine_id, lifetime, lifetime, 0.0, pred, n_star, gamma_star,
                                                    order_event, ordered, order_cycle, arrival_cycle, spare,
                                                    maintenance_event, shortage, decision))
                completed = True
                break

            ratio = pred / mission_cycle
            n_star = int(np.floor(ratio))
            # رابطه (9) مقاله فقط برای n*=1 تعریف شده است.
            gamma_star = float(ratio - np.floor(ratio)) if n_star == 1 else np.nan
            early_recommendation = int(n_star == 1 and gamma_star <= RELIABILITY_THRESHOLD)

            if pred <= maint_threshold:
                maintenance_event = 1; maintenance_cycle = cycle
                maintenance_type = "Predictive maintenance"
                shortage = int(spare == 0); final_shortage = shortage
                decision = "Predictive maintenance" + (" with out-of-stock" if shortage else "")
                decisions.append(locals_to_decision(engine_id, lifetime, cycle, true_rul, pred, n_star, gamma_star,
                                                    order_event, ordered, order_cycle, arrival_cycle, spare,
                                                    maintenance_event, shortage, decision, early_recommendation))
                completed = True
                break

            decision = "Continue operation"
            if not ordered and pred <= ord_threshold:
                ordered = 1; order_event = 1; order_cycle = cycle
                arrival_cycle = int(np.ceil(cycle + lead_time))
                decision = "Order spare part"

            decisions.append(locals_to_decision(engine_id, lifetime, cycle, true_rul, pred, n_star, gamma_star,
                                                order_event, ordered, order_cycle, arrival_cycle, spare,
                                                maintenance_event, shortage, decision, early_recommendation))

        if not completed:
            if ordered and arrival_cycle is not None and lifetime >= arrival_cycle:
                spare = 1
            shortage = int(spare == 0); final_shortage = shortage
            maintenance_cycle = lifetime; maintenance_type = "Corrective maintenance"
            decision = "Corrective maintenance" + (" with out-of-stock" if shortage else "")
            decisions.append(locals_to_decision(engine_id, lifetime, lifetime, 0.0, np.nan, 0, np.nan,
                                                0, ordered, order_cycle, arrival_cycle, spare, 1, shortage, decision))

        summaries.append({
            "engine_id": int(engine_id), "lifetime": lifetime,
            "order_cycle": order_cycle, "arrival_cycle": arrival_cycle,
            "maintenance_cycle": maintenance_cycle, "maintenance_type": maintenance_type,
            "spare_part_status": spare, "out_of_stock": final_shortage,
        })
    return pd.DataFrame(decisions), pd.DataFrame(summaries)


def locals_to_decision(engine_id, lifetime, cycle, true_rul, pred, n_star, gamma_star,
                       order_event, order_status, order_cycle, arrival_cycle, spare,
                       maintenance_event, shortage, decision, early_recommendation=0):
    return {
        "engine_id": int(engine_id), "lifetime": int(lifetime), "cycle": int(cycle),
        "true_rul": float(true_rul), "predicted_rul": float(pred) if np.isfinite(pred) else np.nan,
        "n_star": int(n_star), "gamma_star_when_n1": gamma_star,
        "early_maintenance_recommendation_gamma_le_0_8": int(early_recommendation),
        "order_event": int(order_event), "order_status": int(order_status),
        "order_cycle": order_cycle, "arrival_cycle": arrival_cycle,
        "spare_part_status": int(spare), "maintenance_event": int(maintenance_event),
        "out_of_stock": int(shortage), "decision": decision,
    }


all_decisions_df, engine_summary_df = run_strategy(
    prediction_df, MISSION_CYCLE, delta_value, LEAD_TIME, INSPECTION_INTERVAL
)
all_decisions_df.to_csv(OUTPUT_DIR / "05_all_strategy_decisions.csv", index=False)
engine_summary_df.to_csv(OUTPUT_DIR / "06_engine_strategy_summary.csv", index=False)

strategy_count_df = pd.DataFrame([{
    "number_of_engines": len(engine_summary_df),
    "number_of_orders": int(engine_summary_df.order_cycle.notna().sum()),
    "number_of_predictive_maintenance": int((engine_summary_df.maintenance_type == "Predictive maintenance").sum()),
    "number_of_corrective_maintenance": int((engine_summary_df.maintenance_type == "Corrective maintenance").sum()),
    "number_of_out_of_stock": int(engine_summary_df.out_of_stock.sum()),
}])
strategy_count_df.to_csv(OUTPUT_DIR / "07_strategy_counts.csv", index=False)
display(strategy_count_df)

In [ ]:
# سلول 10: جدول موتورهای منتخب و قابلیت اطمینان مأموریت آخر
selected_engine_ids = [82, 87, 93, 97]
selected_decisions_df = (
    all_decisions_df[all_decisions_df.engine_id.isin(selected_engine_ids)]
    .groupby("engine_id", group_keys=False).tail(6).reset_index(drop=True)
)
selected_decisions_df.to_csv(OUTPUT_DIR / "08_selected_engine_decisions.csv", index=False)

last_mission_reliability_df = all_decisions_df[
    (all_decisions_df.n_star == 1) & all_decisions_df.gamma_star_when_n1.notna()
].copy()
last_mission_reliability_df.to_csv(OUTPUT_DIR / "09_last_mission_reliability.csv", index=False)

display(selected_decisions_df)
display(last_mission_reliability_df.head(20))

In [ ]:
# سلول 11: روابط (11)، (12) و (13) - نرخ هزینه
maintenance_rows = (
    all_decisions_df[all_decisions_df.maintenance_event == 1]
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id", as_index=False).tail(1)
)
cost_df = engine_summary_df.merge(
    maintenance_rows[["engine_id", "predicted_rul", "true_rul"]], on="engine_id", how="left"
)

rows = []
for _, row in cost_df.iterrows():
    shortage_cost = COS if int(row.out_of_stock) else 0.0
    if row.maintenance_type == "Predictive maintenance":
        predicted_lifetime = float(row.maintenance_cycle) + float(row.predicted_rul)
        denominator = predicted_lifetime - (MISSION_CYCLE + delta_value)
        numerator = CP + shortage_cost
    else:
        predicted_lifetime = np.nan
        denominator = float(row.lifetime)
        numerator = CF + shortage_cost
    rows.append({
        **row.to_dict(), "predicted_lifetime": predicted_lifetime,
        "cost_numerator": numerator, "cost_denominator": denominator,
        "proposed_cost_rate": numerator / denominator,
        "ideal_cost_rate": CP / float(row.lifetime),
    })

cost_df = pd.DataFrame(rows).sort_values("engine_id").reset_index(drop=True)
cost_df["group"] = cost_df.index // 5 + 1
group_cost_df = cost_df.groupby("group", as_index=False).agg(
    first_engine=("engine_id", "min"), last_engine=("engine_id", "max"),
    proposed_cost_rate=("proposed_cost_rate", "mean"),
    ideal_cost_rate=("ideal_cost_rate", "mean"),
)
cost_summary_df = pd.DataFrame([{
    "average_proposed_cost_rate": cost_df.proposed_cost_rate.mean(),
    "average_ideal_cost_rate": cost_df.ideal_cost_rate.mean(),
    "relative_gap_percent": 100 * (cost_df.proposed_cost_rate.mean() - cost_df.ideal_cost_rate.mean()) / cost_df.ideal_cost_rate.mean(),
    "paper_proposed_reference": 0.491,
    "paper_ideal_reference": 0.476,
}])

cost_df.to_csv(OUTPUT_DIR / "10_engine_cost_rates.csv", index=False)
group_cost_df.to_csv(OUTPUT_DIR / "11_group_cost_rates.csv", index=False)
cost_summary_df.to_csv(OUTPUT_DIR / "12_cost_summary.csv", index=False)
display(cost_summary_df)
display(group_cost_df)

In [ ]:
# سلول 12: نمودار RUL واقعی و پیش‌بینی تجمیعی چهار موتور
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), constrained_layout=True)
for ax, engine_id, panel in zip(axes.ravel(), selected_engine_ids, ["(a)", "(b)", "(c)", "(d)"]):
    e = prediction_df[prediction_df.engine_id == engine_id].sort_values("cycle")
    ax.plot(e.cycle, e.true_rul_capped, "--", color="#005F73", linewidth=1.8, label="Actual RUL")
    ax.plot(e.cycle, e.predicted_rul, "-", color="#D62828", linewidth=1.5, label="Predicted RUL")
    summary = engine_summary_df[engine_summary_df.engine_id == engine_id].iloc[0]
    if pd.notna(summary.order_cycle): ax.axvline(summary.order_cycle, color="#F4A261", linestyle=":", label="Order")
    ax.axvline(summary.maintenance_cycle, color="#2A9D8F", linestyle="-.", label="Maintenance")
    ax.set_title(f"{panel} Turbofan engine ID{engine_id}")
    ax.set_xlabel("Operating cycle"); ax.set_ylabel("RUL (cycles)")
    ax.set_ylim(0, 140); ax.grid(alpha=0.25); ax.legend(fontsize=8)
plt.savefig(OUTPUT_DIR / "03_four_engines_rul_and_decisions.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# سلول 13: نمودار مقایسه نرخ هزینه گروه‌ها و هر موتور
labels = [f"ID{int(r.first_engine)}-{int(r.last_engine)}" for _, r in group_cost_df.iterrows()]
xpos = np.arange(len(labels)); width = 0.35
plt.figure(figsize=(9, 5.5))
plt.bar(xpos-width/2, group_cost_df.proposed_cost_rate, width, color="#2A9D8F", label="Implemented strategy")
plt.bar(xpos+width/2, group_cost_df.ideal_cost_rate, width, color="#7B2CBF", label="Ideal strategy")
plt.xticks(xpos, labels); plt.ylabel("Average maintenance cost rate")
plt.title("Maintenance cost-rate comparison")
plt.grid(axis="y", alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_group_cost_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(11, 5.5))
plt.plot(cost_df.engine_id, cost_df.proposed_cost_rate, "o-", label="Implemented strategy", color="#2A9D8F")
plt.plot(cost_df.engine_id, cost_df.ideal_cost_rate, "s--", label="Ideal strategy", color="#7B2CBF")
plt.xlabel("Engine ID"); plt.ylabel("Maintenance cost rate")
plt.title("Cost rate for each evaluation engine")
plt.grid(alpha=0.25); plt.legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_engine_cost_rates.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# سلول 14: تحلیل حساسیت چرخه مأموریت و آستانه δ
mission_cycles = [2.940, 3.770, 4.550, 5.300]
sensitivity_rows = []
for dt in mission_cycles:
    ddf, sdf = run_strategy(prediction_df, dt, delta_value, LEAD_TIME, INSPECTION_INTERVAL)
    sensitivity_rows.append({
        "mission_cycle": dt,
        "orders": int(sdf.order_cycle.notna().sum()),
        "predictive_maintenance": int((sdf.maintenance_type == "Predictive maintenance").sum()),
        "corrective_maintenance": int((sdf.maintenance_type == "Corrective maintenance").sum()),
        "out_of_stock": int(sdf.out_of_stock.sum()),
    })
mission_sensitivity_df = pd.DataFrame(sensitivity_rows)
mission_sensitivity_df.to_csv(OUTPUT_DIR / "13_mission_cycle_sensitivity.csv", index=False)

threshold_rows = []
for label, delta_used in [("Delta_t only", 0.0), ("Delta_t plus delta", delta_value)]:
    ddf, sdf = run_strategy(prediction_df, MISSION_CYCLE, delta_used, LEAD_TIME, INSPECTION_INTERVAL)
    threshold_rows.append({
        "threshold_case": label,
        "delta_used": delta_used,
        "predictive_maintenance": int((sdf.maintenance_type == "Predictive maintenance").sum()),
        "corrective_maintenance": int((sdf.maintenance_type == "Corrective maintenance").sum()),
        "out_of_stock": int(sdf.out_of_stock.sum()),
    })
threshold_sensitivity_df = pd.DataFrame(threshold_rows)
threshold_sensitivity_df.to_csv(OUTPUT_DIR / "14_threshold_sensitivity.csv", index=False)
display(mission_sensitivity_df)
display(threshold_sensitivity_df)

In [ ]:
# سلول 15: فایل توضیحات روش و بسته نهایی نتایج
method_notes = (
    "FD001 dynamic predictive maintenance results\n\n"
    "Prediction model configuration follows the authors' released code:\n"
    f"MAX_RUL={MAX_RUL}, WINDOW_SIZE={WINDOW_SIZE}, MinMax scaling, 14 sensors,\n"
    f"Conv1D(64,kernel=3,ReLU), Dropout={DROPOUT_RATE}, BiLSTM(64,concat),\n"
    f"second Dropout={DROPOUT_RATE}, Flatten, linear Dense, Adam(lr={LEARNING_RATE}),\n"
    f"batch={BATCH_SIZE}, epochs={EPOCHS}, runs={N_RUNS}.\n\n"
    "Maintenance equations follow Wang et al. (2024), Eqs. (6)-(14).\n"
    "Engines 1-80 are used for training and engines 81-100 for cycle-wise evaluation.\n"
    "Delta is computed, article-faithfully, on evaluation samples with true RUL < 30.\n"
    "The gamma-star <= 0.8 rule is reported as an auxiliary early-maintenance recommendation\n"
    "and is not mixed into the primary Eq. (6) cost calculation.\n\n"
    "Important article ambiguity: page 8 prints Cp=100 and Cf=100, while Section 4.4\n"
    "explicitly uses Cp=100, Cf=500, Cos=10 for cost comparison. This notebook uses\n"
    "the Section 4.4 values because they generate the reported comparison framework.\n"
)
(OUTPUT_DIR / "README_method_notes.txt").write_text(method_notes, encoding="utf-8")

zip_path = Path("/content/FD001_dynamic_maintenance_all_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(OUTPUT_DIR))

print("Final results package:", zip_path)
print("Number of files:", len([p for p in OUTPUT_DIR.rglob('*') if p.is_file()]))
files.download(str(zip_path))

## فهرست خروجی‌ها

- خلاصه و جزئیات اجرای مدل‌ها
- فایل checkpoint پیش‌بینی‌ها
- جدول δ و پارامترهای استراتژی
- تمام تصمیم‌های چرخه‌ای ۲۰ موتور
- خلاصه سفارش، موجودی و نوع نگهداری هر موتور
- جدول چهار موتور منتخب
- جدول قابلیت اطمینان مأموریت آخر
- هزینه هر موتور، میانگین گروه‌ها و خلاصه هزینه
- تحلیل حساسیت چرخه مأموریت و آستانه δ
- نمودار Loss، توزیع گاما، RUL چهار موتور و نرخ هزینه
- فایل متنی ثبت تنظیمات و فرض‌های اجرایی

فایل ZIP نهایی تمام این موارد را یکجا دانلود می‌کند.